# Image CLI Optimizer

This notebook accompanies Chapter 10 §10.2.1–10.2.3 of *Context Engineering with DSPy*: building and optimizing the image-population CLI that fills the placeholder image slots emitted by the landing-page skill from §10.1.

The CLI is a four-predictor `dspy.Module` — extract the image briefs from the page HTML, write an image-generation prompt for each one, generate the image, score it against the brand. The whole thing gets optimized with `dspy.GEPA`, where every predictor gets its own slot for reflection-driven mutation.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used for both the chat LM (`openai/gpt-5-mini`), the strong reflection LM (`openai/gpt-5.5-pro`), and the image LM (`openai/gpt-image-2-mini`).
- `ANTHROPIC_API_KEY` — used by Claude Code CLI when invoked as the agent under optimization elsewhere in this chapter. Not strictly required for this notebook, but keep it set for parity with the rest of §10.

## Required tooling

The **Claude Code CLI** must be installed if you want to invoke the optimized CLI from inside a coding-agent session (this is how the chapter actually deploys it):

```bash
npm install -g @anthropic-ai/claude-code
```

## Required assets

The seed-failure dataset references `acme_landing.html` and `outpost_landing.html` — pages produced by the landing-page skill from §10.1. Drop your own under `./assets/landing-pages/`. We don't ship sample HTML in the repo because the dataset only does its job when it's *your* real failed pages.


In [ ]:
%pip install -r ../requirements.txt -q

## Setup

Load env vars and configure DSPy. We default to `openai/gpt-5-mini` for the text predictors — counterintuitively, optimizing on a cheap model usually transfers to the strong one at deploy time.


In [ ]:
import os, dspy
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not in env"

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))


## Step 1: four signatures, one module

Each predictor gets its own `dspy.Signature`, which means each gets its own slot for GEPA to mutate, and each gets its own `pred_name` when the reflection LM is deciding what to fix. Per-predictor credit assignment is the payoff for breaking a CLI into a real module instead of one giant prompt.


In [ ]:
class ExtractImageBriefs(dspy.Signature):
    """Pull the image slots out of the page HTML.

    For each <img> with a data-image-brief attribute, return the brief,
    the surrounding section's purpose (hero, feature, testimonial), and
    the aspect ratio the layout expects.
    """
    page_html: str = dspy.InputField()
    slots: list[dict] = dspy.OutputField(
        desc="List of {slot_id, brief, section_role, aspect_ratio}"
    )


class WriteImagePrompt(dspy.Signature):
    """Turn a single image brief + the brand guide into a generation
    prompt for the image model. Emphasize the brand palette, mood, and
    style. Avoid asking for text inside the image."""
    brief:       str  = dspy.InputField()
    brand_guide: dict = dspy.InputField()
    section_role: str = dspy.InputField()
    image_prompt: str = dspy.OutputField()


class ScoreImageAgainstBrand(dspy.Signature):
    """Judge whether the generated image matches the brand guide.
    Return a score and concrete feedback the next attempt can act on."""
    brand_guide:   dict        = dspy.InputField()
    brief:         str         = dspy.InputField()
    image:         dspy.Image  = dspy.InputField()
    score:         float       = dspy.OutputField(desc="0.0 to 1.0")
    feedback:      str         = dspy.OutputField(
        desc="What's off about this image and how to fix it"
    )


class CLIMetric(dspy.Signature):
    """Composite metric for the whole image CLI run.

    Compare the filled page against the brief and brand guide. Each
    filled slot contributes to the score. Penalize stock-photo vibes,
    color-cast mismatches, and any image with embedded text.
    """
    brief:           str  = dspy.InputField()
    brand_guide:     dict = dspy.InputField()
    filled_slots:    dict = dspy.InputField(desc="slot_id -> image_url")
    score:           float = dspy.OutputField(desc="0.0 to 1.0")
    feedback:        str   = dspy.OutputField(
        desc="Per-slot critique the optimizer can act on"
    )


## Step 2: the `ImagePopulator` module

The inner `dspy.Refine` is the move that makes this work — it runs `_generate_one` up to three times, keeps the best by score, and feeds the prior attempt's feedback into the next attempt's context. Generator, judge, retry. The judge's reasoning *and* the generator's prompt are both first-class signatures GEPA can mutate per-predictor.


In [ ]:
class ImagePopulator(dspy.Module):
    """Fill the placeholder image slots in a landing page."""
    def __init__(self, image_model="openai/gpt-image-2"):
        super().__init__()
        self.extract       = dspy.ChainOfThought(ExtractImageBriefs)
        self.write_prompt  = dspy.ChainOfThought(WriteImagePrompt)
        self.score         = dspy.Predict(ScoreImageAgainstBrand)
        self.image_lm      = dspy.LM(image_model)

    def forward(self, page_html, brand_guide):
        slots = self.extract(page_html=page_html).slots
        filled = {}
        for slot in slots:
            # dspy.Refine wraps generate + score into a self-correcting loop.
            refined = dspy.Refine(
                module=self._generate_one,
                N=3,
                reward_fn=lambda pred: pred.score,
                threshold=0.8,
            )
            result = refined(slot=slot, brand_guide=brand_guide)
            filled[slot["slot_id"]] = result.image
        return dspy.Prediction(filled_slots=filled)

    def _generate_one(self, slot, brand_guide):
        prompt = self.write_prompt(
            brief=slot["brief"], brand_guide=brand_guide,
            section_role=slot["section_role"],
        ).image_prompt
        image = self.image_lm(prompt=prompt, size=slot["aspect_ratio"])
        judgment = self.score(brand_guide=brand_guide, brief=slot["brief"], image=image)
        return dspy.Prediction(image=image, score=judgment.score,
                               feedback=judgment.feedback)


## Step 3: CLI seed failures

These are *image-CLI* failures, not page-layout failures — the SKILL.md was fine, the images it slotted weren't. Note them as you hit them: hero filled with a generic stock-photo, color cast that doesn't match the brand, an image with embedded text the model couldn't spell. The references to `acme_landing.html` etc. should point at your own pages.


In [ ]:
cli_seed_failures = [
    {
        "page_html":   open("./assets/landing-pages/acme_landing.html").read(),
        "brand_guide": {"colors": {"primary": "#4338CA"}, "mood": "minimal"},
        "failure_mode": "stock_photo_vibes",
        "notes": "Hero filled with a generic shaking-hands photo. Brand "
                 "guide said minimal/abstract.",
    },
    {
        "page_html":   open("./assets/landing-pages/outpost_landing.html").read(),
        "brand_guide": {"colors": {"primary": "#2D5016"}, "mood": "warm, human"},
        "failure_mode": "color_cast_off",
        "notes": "Feature images had a blue cast; brand is forest green.",
    },
    # ... 3-8 more like this.
]


## Step 4: expand the dataset synthetically

CLI-layer rollouts are cheap (direct LM + image calls, no agent subprocess), so we can afford to grow the dataset by an order of magnitude. A `SyntheticImageTaskGenerator` signature takes the seed failures and generates more `(page_html, brand_guide, gold_filled_slots)` triples that target the same kinds of image-fit failures.


In [ ]:
class SyntheticImageTaskGenerator(dspy.Signature):
    """Given real failed runs of the image-population CLI, generate
    additional page + brand-guide pairs that target the same kinds of
    image-fit failures.

    For each generated task, hand-pick a reference image style from the
    gold set so the visual judge has something to compare against.
    """
    seed_failures: list[dict] = dspy.InputField(
        desc="Real failed runs: page_html, brand_guide, what went wrong"
    )
    num_to_generate: int = dspy.InputField()

    new_tasks: list[dict] = dspy.OutputField(
        desc="List of {page_html, brand_guide, gold_filled_slots, "
             "failure_mode} dicts"
    )


generate = dspy.ChainOfThought(SyntheticImageTaskGenerator)
synthetic = generate(seed_failures=cli_seed_failures,
                     num_to_generate=50).new_tasks

trainset = [dspy.Example(**t).with_inputs("page_html", "brand_guide")
            for t in cli_seed_failures + synthetic]

# Hold a small valset back from the trainset for GEPA's pareto frontier.
valset   = trainset[-10:]
trainset = trainset[:-10]


## Step 5: optimize on cheap models, deploy on strong ones

`dspy.context` lets us swap in the cheap chat and image models for the optimization pass. The reflection LM stays strong — `openai/gpt-5.5-pro` — because its job is to *propose* better prompts, and that's the one place weak models really fall down.

This cell uses `max_metric_calls=3` to smoke-test wiring. The next cell is the full-budget version (commented out) with `max_metric_calls=600` — sensible at the CLI layer because every rollout is cents, not dollars.


In [ ]:
def cli_metric(example, prediction, trace=None):
    """Lightweight metric stub. Replace with your real scoring logic —
    e.g. ScoreImageAgainstBrand averaged across slots, or a pairwise
    judge against gold_filled_slots."""
    slots = getattr(prediction, "filled_slots", None) or {}
    return float(len(slots) > 0)


with dspy.context(lm=dspy.LM("openai/gpt-5-mini"),
                  image_lm=dspy.LM("openai/gpt-image-2-mini")):
    optimizer = dspy.GEPA(
        metric=cli_metric,
        reflection_lm=dspy.LM("openai/gpt-5.5-pro",
                              temperature=1.0, max_tokens=8000),
        max_metric_calls=3,
        reflection_minibatch_size=2,
        candidate_selection_strategy="pareto",
        num_threads=2,
        track_stats=True,
        log_dir="./gepa_logs",
    )
    optimized = optimizer.compile(
        student=ImagePopulator(),
        trainset=trainset,
        valset=valset,
    )

print("done — best score:", getattr(optimized, "score", None))


In [ ]:
# Full reproduction of the book's §10.3.5 run.
# Uncomment to run — ~600 rollouts at cents each, on the order of $10-$30.
#
# with dspy.context(lm=dspy.LM("openai/gpt-5-mini"),
#                   image_lm=dspy.LM("openai/gpt-image-2-mini")):
#     optimizer = dspy.GEPA(
#         metric=cli_metric,
#         reflection_lm=dspy.LM("openai/gpt-5.5-pro",
#                               temperature=1.0, max_tokens=8000),
#         max_metric_calls=600,
#         reflection_minibatch_size=8,
#         candidate_selection_strategy="pareto",
#         num_threads=6,
#         track_stats=True,
#         log_dir="./gepa_logs",
#     )
#     optimized = optimizer.compile(
#         student=ImagePopulator(),
#         trainset=trainset,
#         valset=valset,
#     )
#
# # Persist the optimized prompts so Claude Code (and the landing-page
# # skill from §10.1) gets the better version on the next invocation.
# optimized.save("image_populator_optimized.json")
#
# # At deploy time, swap in the strong models. The optimized prompts transfer.
# production = ImagePopulator(image_model="openai/gpt-image-2")
# production.load_state(optimized.dump_state())


The optimized CLI inherits the landing-page skill's improvements for free: the skill emits placeholders, the CLI fills them, and the page that ships to the user has been improved at two layers in the same optimization pass.
